# Juliet frozen Test 진단 및 전체 평가

기본값은 leakage-disjoint frozen Test의 결정적 500건 진단입니다. 진단을 통과한 뒤 `DIAGNOSTIC_CASE_LIMIT=0`으로 바꾸면 8,337개 전체에서 LR·GBDT·Multi-task MLP Router를 같은 outcome matrix로 비교합니다.

In [ ]:
from pathlib import Path

EVAL_ROOT = Path.cwd().resolve()
if EVAL_ROOT.name != 'Model_Evaluation':
    EVAL_ROOT = (EVAL_ROOT / 'Model_Evaluation').resolve()
CONFIG_PATH = EVAL_ROOT / 'configs' / 'full.toml'
COHORT_CONFIG_PATH = EVAL_ROOT / 'configs' / 'cohort_15837.toml'
ENV_FILE = EVAL_ROOT.parent / '.env'
SELECTED_ARTIFACT = EVAL_ROOT / 'artifacts' / 'juliet_utility_router.pkl'
CANDIDATE_RANKER_ARTIFACT = EVAL_ROOT / 'artifacts' / 'candidate_ranker' / 'candidate_ranker.pkl'
MAX_CANDIDATES_PER_CASE = 4
HARD_NEGATIVES_PER_CASE = 1
MAX_CONCURRENCY = 1000  # OpenRouter 한도에 맞춰 낮출 수 있음
DIAGNOSTIC_CASE_LIMIT = 0  # 0으로 바꿔야 frozen Test 전체를 실행합니다.
RUN_PATCH_EVALUATION = False
PATCH_CASE_LIMIT = 0


In [ ]:
import json, shutil, sys
from itertools import islice
sys.path.insert(0, str(EVAL_ROOT / 'src'))
from model_evaluation.config import load_config, load_mapping
from model_evaluation.stages.select_cohort import load_cohort_config, ensure_frozen_index, build_cohort_manifests
from model_evaluation.stages.materialize_dataset import materialize_dataset
from model_evaluation.candidates import cache_candidates
from model_evaluation.candidate_ranking import rank_candidate_cache
from model_evaluation.jsonl import iter_jsonl, write_jsonl
from model_evaluation.workflow import plan_outcome_matrix, collect_outcome_matrix, audit_outcome_matrix, evaluate_utility_router
from model_evaluation.live_evaluation import run_batched_patch_evaluation
from model_evaluation.adapters.llm_security import activate_parent_package
activate_parent_package()
from llm_security.routing import BudgetedUtilityRouter

config = load_config(CONFIG_PATH)
mapping = load_mapping(config.paths.mapping)
cohort_config = load_cohort_config(COHORT_CONFIG_PATH)
COHORT_DIR = EVAL_ROOT / 'work' / 'cohort_15837'
RUN_NAME = ('router_evaluation_diagnostic_' + str(DIAGNOSTIC_CASE_LIMIT) if DIAGNOSTIC_CASE_LIMIT else 'router_evaluation_full_test')
RUN_DIR = EVAL_ROOT / 'work' / RUN_NAME
RESULT_DIR = EVAL_ROOT / 'results' / RUN_NAME
TRAINING_REPORT = EVAL_ROOT / 'results' / 'router_training_stratified_7500' / 'training_report.json'
RUN_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)
if not SELECTED_ARTIFACT.exists() or not CANDIDATE_RANKER_ARTIFACT.exists() or not TRAINING_REPORT.exists():
    raise FileNotFoundError('train.ipynb의 Router suite 학습을 먼저 완료하세요.')
training_report = json.loads(TRAINING_REPORT.read_text(encoding='utf-8'))
validator_thresholds = training_report.get('validator_minimum_confidence_by_expert', {})
artifacts = {name: Path(row['artifact']) for name, row in training_report['variants'].items()}
selected_router = BudgetedUtilityRouter.load(SELECTED_ARTIFACT)
models = sorted({item.model_id for item in selected_router.assignments.values()})
if len(models) != 1:
    raise ValueError('Outcome matrix는 하나의 physical model을 사용해야 합니다.')
print('Selected backend:', training_report['selected_backend'])
print('Physical model:', models[0])

## 1. Frozen Test 진단/전체 manifest, materialization, candidate cache

In [ ]:
index_report = ensure_frozen_index(config, mapping, progress=print)
cohort_report = build_cohort_manifests(config, cohort_config, output_directory=COHORT_DIR)
full_test_manifest = COHORT_DIR / 'cohort_test.jsonl'
if DIAGNOSTIC_CASE_LIMIT:
    test_manifest = RUN_DIR / 'cohort_test_diagnostic.jsonl'
    write_jsonl(test_manifest, islice(iter_jsonl(full_test_manifest), DIAGNOSTIC_CASE_LIMIT))
else:
    test_manifest = full_test_manifest
materialization = materialize_dataset(
    config, mapping, output_directory=RUN_DIR / 'cases', splits=('test',),
    selection_manifests={'test': test_manifest}, progress=print,
)
candidate_summary = cache_candidates(
    RUN_DIR / 'cases' / 'cases_test.jsonl',
    RUN_DIR / 'candidates' / 'candidates_test.jsonl',
    max_source_bytes=config.max_source_bytes, parse_timeout_ms=config.parse_timeout_ms, progress=print,
)
candidate_cache_path = RUN_DIR / 'candidates' / 'candidates_test_ranked.jsonl'
candidate_ranker_summary = rank_candidate_cache(
    cases_path=RUN_DIR / 'cases' / 'cases_test.jsonl',
    input_cache=RUN_DIR / 'candidates' / 'candidates_test.jsonl',
    output_path=candidate_cache_path, artifact_path=CANDIDATE_RANKER_ARTIFACT,
)
print(json.dumps({'cohort': cohort_report['splits']['test'], 'materialization': materialization, 'candidates': candidate_summary, 'candidate_ranker': candidate_ranker_summary}, ensure_ascii=False, indent=2))

## 2. Full-5 measured outcome matrix 수집 (기본 500건 진단, case당 API 최대 1회)

In [ ]:
selection_manifest = RUN_DIR / 'selections' / 'selected_test.jsonl'
outcome_path = RUN_DIR / 'outcomes' / 'outcomes_test.jsonl'
plan = plan_outcome_matrix(
    cases_path=RUN_DIR / 'cases' / 'cases_test.jsonl',
    candidate_cache=candidate_cache_path,
    selection_manifest=selection_manifest, outcome_path=outcome_path, model_ids=models,
    selection_policy='deployment_top_k', max_candidates_per_case=MAX_CANDIDATES_PER_CASE,
    hard_negatives_per_case=0,
)
print(json.dumps(plan, ensure_ascii=False, indent=2))
collection = collect_outcome_matrix(
    env_file=ENV_FILE, cases_path=RUN_DIR / 'cases' / 'cases_test.jsonl',
    candidate_cache=candidate_cache_path,
    outcome_path=outcome_path, ledger_path=RUN_DIR / 'ledgers' / 'test_api_ledger.jsonl',
    selection_manifest=selection_manifest, model_ids=models,
    selection_policy='deployment_top_k', max_candidates_per_case=MAX_CANDIDATES_PER_CASE,
    max_concurrency=MAX_CONCURRENCY,
    validator_minimum_confidence_by_expert=validator_thresholds,
)
print(json.dumps(collection, ensure_ascii=False, indent=2))
if collection['status'] != 'complete':
    print('실패한 case만 남았습니다. 성공한 case는 저장되었으며 이 셀을 다시 실행하면 실패 case부터 재개합니다.')

## 3. LR / GBDT / MLP 동일 Test 비교

In [ ]:
expected_ids = list(selected_router.assignments)
audit = audit_outcome_matrix(outcome_path, expected_assignment_ids=expected_ids, selection_manifest=selection_manifest) if outcome_path.exists() else {'complete': False}
print(json.dumps(audit, ensure_ascii=False, indent=2))
evaluation_reports = {}
if audit['complete']:
    for backend, artifact in artifacts.items():
        evaluation_reports[backend] = evaluate_utility_router(
            artifact_path=artifact, test_outcomes=outcome_path,
            test_cases=RUN_DIR / 'cases' / 'cases_test.jsonl',
            candidate_cache=candidate_cache_path,
            selection_manifest=selection_manifest,
            report_path=RESULT_DIR / f'evaluation_{backend}.json',
            max_candidates_per_case=MAX_CANDIDATES_PER_CASE,
        )
    print(json.dumps(evaluation_reports, ensure_ascii=False, indent=2))
else:
    print('Test outcome matrix가 아직 완성되지 않았습니다. 2번 셀을 다시 실행하세요.')

## 4. 선택된 Router의 GT-matched finding 일괄 패치 평가

In [ ]:
if RUN_PATCH_EVALUATION and audit.get('complete'):
    verifier_commands = ([{'name': 'juliet-build', 'command': ['make'], 'timeout_seconds': 300.0}] if shutil.which('make') else [])
    patch_report = run_batched_patch_evaluation(
        env_file=ENV_FILE, detection_path=outcome_path.with_suffix('.detections.jsonl'),
        output_path=RUN_DIR / 'patches' / 'patch_results.jsonl',
        ledger_path=RUN_DIR / 'ledgers' / 'patch_api_ledger.jsonl',
        commands=verifier_commands, max_cases=PATCH_CASE_LIMIT,
    )
    patch_report['verification_mode'] = 'compile/test' if verifier_commands else 'apply-and-diff-check-only'
    (RESULT_DIR / 'patch_report.json').write_text(json.dumps(patch_report, ensure_ascii=False, indent=2), encoding='utf-8')
    print(json.dumps(patch_report, ensure_ascii=False, indent=2))